## 1. Data loading & preprocessing

In [140]:
import pandas as pd
import numpy as np
import math

### Normalising data



In [141]:
df = pd.read_csv("Train.csv").fillna(0)

In [142]:
labels = df["label"]
labels

0        1
1        0
2        1
3        4
4        0
        ..
41995    0
41996    1
41997    7
41998    6
41999    9
Name: label, Length: 42000, dtype: int64

In [143]:
df = df.drop("label",axis=1)
df = df.apply(lambda x: x/255) #This is done so each pixel value of 0-255 is between 0 and 1, which is computationally faster in multiplication
df = df.T
df

,0,1,2,3,4,5,6,7,8,9,...,41990,41991,41992,41993,41994,41995,41996,41997,41998,41999
pixel0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
pixel779,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel780,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel781,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pixel782,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Hotshotting labels and defining parameters
Here, we take the rows of I10 and the label corresponds to the location of the 1 in the I10 row. ex 5 = [0,0,0,0,1,0,0,0,0,0]

In [144]:
Y = np.eye(10)[:,labels]
Y

array([[0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(10, 42000))

In [ ]:
input_layer_size = df.shape[0] #784
hidden_layer_size = 128
training_data_size = df.shape[1] #42000
alpha = 0.01
training_steps = 500


## Forward propogation


TODO: Write calculations down

In [146]:
# Assumes your df is (42000, 784). Transpose it to make it (784, 42000)
a0 = np.array(df)

input_layer_size = 784
hidden_layer_size = 128
training_data_size = a0.shape[1] # 42000

# He (Kaiming) Initialization - Done ONCE before training starts
std1 = math.sqrt(2 / input_layer_size)
std2 = math.sqrt(2 / hidden_layer_size)

w1 = std1 * np.random.randn(hidden_layer_size, input_layer_size)
b1 = np.zeros((hidden_layer_size, 1))

w2 = std2 * np.random.randn(10, hidden_layer_size)
b2 = np.zeros((10, 1))

def forward_propagate(a0, w1, b1, w2, b2):
    # Hidden Layer
    z1 = np.dot(w1, a0) + b1
    a1 = np.maximum(0, z1) # ReLU

    # Output Layer
    z2 = np.dot(w2, a1) + b2

    # Stable Softmax (Subtract column-wise max to prevent overflow)
    exp_z2 = np.exp(z2 - np.max(z2, axis=0, keepdims=True))
    a2 = exp_z2 / np.sum(exp_z2, axis=0, keepdims=True)
    
    return z1, a1, z2, a2

z1, a1, z2, a2 = forward_propagate(a0, w1, b1, w2, b2)

## Backpropogation


### Loss function


In [147]:
def cross_entropy_loss(Y,output_matrix):
    return (np.sum(Y * np.log(output_matrix) * -1) / training_data_size)

cross_entropy_loss(Y,a2)

np.float64(2.3895022133661454)

In [148]:
def back_propagate(w1, b1, w2, b2, z1, a0, a1, a2, Y, training_data_size, alpha):
    # 1. Output Layer Gradients (Layer 2)
    dz2 = (1 / training_data_size) * (a2 - Y)
    dW2 = np.dot(dz2, a1.T)
    db2 = np.sum(dz2, axis=1, keepdims=True)

    # 2. Hidden Layer Gradients (Layer 1)
    dz1 = np.dot(w2.T, dz2) * (z1 > 0)
    dW1 = np.dot(dz1, a0.T)
    db1 = np.sum(dz1, axis=1, keepdims=True)

    # 3. Parameter Updates (Gradient Descent)
    w2 = w2 - alpha * dW2
    b2 = b2 - alpha * db2
    w1 = w1 - alpha * dW1
    b1 = b1 - alpha * db1
    
    # Return everything back to the training loop
    return w1, b1, w2, b2

### Training loop


In [154]:
def training_loop(training_steps, a0, Y, w1, b1, w2, b2, training_data_size, alpha):
   for step in range(training_steps):

      # Forward propogate:
      z1, a1, _, a2 = forward_propagate(a0, w1, b1, w2, b2)

      # Back propogate
      w1, b1, w2, b2 = back_propagate(
            w1, b1, w2, b2, z1, a0, a1, a2, Y, training_data_size, alpha
        )

      # 3. Housekeeping: Print progress every 100 steps
      if step % 100 == 0:
         # Calculate Cross-Entropy Loss
         # (the 1e-15 prevents a log(0) crash if a2 hits exactly 0)
         loss = -np.sum(Y * np.log(a2 + 1e-15)) / training_data_size
         
         # Calculate Accuracy
         predictions = np.argmax(a2, axis=0)
         true_labels = np.argmax(Y, axis=0) # Assumes Y is one-hot encoded
         accuracy = np.mean(predictions == true_labels) * 100
         
         print(f"Step {step:4d} | Loss: {loss:.4f} | Training Accuracy: {accuracy:.2f}%")
   
   return a2

    



In [155]:
if __name__ == "__main__":
    outputs = training_loop(
        training_steps, a0, Y, w1, b1, w2, b2, training_data_size, alpha
    )
    final_predictions = np.argmax(outputs, axis=0)


Step    0 | Loss: 2.3895 | Training Accuracy: 11.53%
Step  100 | Loss: 1.5433 | Training Accuracy: 64.96%
Step  200 | Loss: 1.0881 | Training Accuracy: 77.48%
Step  300 | Loss: 0.8495 | Training Accuracy: 81.95%
Step  400 | Loss: 0.7147 | Training Accuracy: 84.04%


In [157]:
labels_vs_predictions = pd.DataFrame({
    "Label" : labels,
    "Prediction" : final_predictions
})

labels_vs_predictions.head(100)

,Label,Prediction
0,1,1
1,0,0
2,1,1
3,4,2
4,0,0
...,...,...
95,9,7
96,1,1
97,2,2
98,0,0
